In [17]:
import ollama # type: ignore

MODEL="qwen2.5-coder:14b"

In [18]:
def call_model(messages):
    response=ollama.chat(
        model=MODEL,
        messages=messages,
        options={"temperature":0.1}
    )
    return response['message']['content']

In [19]:
test_system_prompt="""
You are an expert database test data generator.

Your task is to generate SQL INSERT statements for the given schema.

STRICT RULES:
- Output ONLY SQL (no explanation, no markdown, no ```sql)
- Do NOT include any extra text
- Use exact table and column names from schema
- Maintain referential integrity (valid foreign keys)
- Respect primary keys (no duplicates)
- Respect NOT NULL constraints

DATA RULES:
- Generate 3-5 rows per table
- Use realistic values (names, emails, etc.)
- Ensure foreign keys match parent tables

OUTPUT:
- Only INSERT INTO statements
- Clean, executable SQL
"""

In [20]:
def read_sql_file(filepath="schema.sql"):
    with open(filepath,"r",encoding="utf-8") as f:
        return f.read()

In [21]:
def clean_sql_output(text):
    text=text.strip()

    if text.startswith("```"):
        parts=text.split("```")
        if len(parts)>=2:
            text=parts[1].strip()

    if text.lower().startswith("sql"):
        text=text[3:].strip()

    return text

In [22]:
def generate_test_data_from_file(sql_file="schema.sql",max_retries=3):
    sql_schema=read_sql_file(sql_file)

    messages=[
        {"role":"system","content":test_system_prompt},
        {
            "role":"user",
            "content":f"""
                    Generate SQL INSERT statements for this schema:

                    {sql_schema}

                    Return ONLY INSERT INTO statements.
                    """
        }
    ]

    for attempt in range(max_retries):
        print(f"\nTest Data Generation Attempt {attempt+1}")

        output=clean_sql_output(call_model(messages))

        if "INSERT INTO" in output.upper():
            print("Test data generated successfully!")
            return output

        messages.append({"role":"assistant","content":output})
        messages.append({
            "role":"user",
            "content":"""
                    The output is not valid SQL.

                    Return ONLY INSERT INTO statements.
                    No markdown. No explanation.
                    """
        })

    raise ValueError("Failed to generate valid test data")

In [23]:
def save_test_data(sql_text,output_file="test_data.sql"):
    with open(output_file,"w",encoding="utf-8") as f:
        f.write(sql_text)

    print(f"\nTest data saved to {output_file}")

In [24]:
print("\nGenerating Test Data\n")

test_data_sql=generate_test_data_from_file("data/Schema.sql")
print(test_data_sql)

save_test_data(test_data_sql,output_file="data/Test_data.sql")


Generating Test Data


Test Data Generation Attempt 1
Test data generated successfully!
INSERT INTO Student (student_id, first_name, last_name, email) VALUES 
(1, 'John', 'Doe', 'john.doe@example.com'),
(2, 'Jane', 'Smith', 'jane.smith@example.com'),
(3, 'Alice', 'Johnson', 'alice.johnson@example.com');

INSERT INTO Teacher (teacher_id, first_name, last_name, email) VALUES 
(101, 'Emily', 'Davis', 'emily.davis@example.com'),
(102, 'Michael', 'Brown', 'michael.brown@example.com');

INSERT INTO Course (course_id, course_name, description) VALUES 
(201, 'Mathematics', 'Advanced math topics'),
(202, 'Science', 'Basic science concepts'),
(203, 'History', 'World history overview');

INSERT INTO Enrollment (enrollment_id, student_id, course_id) VALUES 
(301, 1, 201),
(302, 2, 202),
(303, 3, 203);

INSERT INTO Attendance (attendance_id, student_id, course_id, date, present) VALUES 
(401, 1, 201, '2023-10-01', TRUE),
(402, 2, 202, '2023-10-01', FALSE),
(403, 3, 203, '2023-10-01', TRUE);

INSER